# 00 — Project Overview: ORME/PGM Virtual Lab

This notebook is the guided tour. It explains what the project is, runs a screen end-to-end, and shows how to read the output.

**Framing (read this first):** we take the ORME/high-spin PGM superconductivity premise as a *working assumption* to reverse-engineer how it could be realized. But the validation layer must be able to *fail* — so every score here is a **triage signal**, never a proof. The superconductivity gate can only ever report `NOT RULED OUT`, never `PROVEN`.

See `docs/hypothesis_matrix.md`, `docs/simulation_pipeline.md`, `docs/validation_tests.md`, and `docs/terminology_translation.md` for the science.

In [ ]:
# Make the src/ package importable when running from the notebooks/ dir.
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))

import orme_lab
print('orme_lab version:', orme_lab.__version__)

## The candidate space

A **candidate** is the triple `(element x geometry x spin state)`. The six spec elements are Au, Pt, Pd, Ir, Rh, Os.

In [ ]:
from orme_lab.elements import core_screen_elements
for e in core_screen_elements():
    print(f'{e.symbol:2}  Z={e.atomic_number:3}  d={e.d_electrons} s={e.s_electrons}  {e.valence_config}')

## Run a screen

`run_screen()` enumerates candidates, scores each through the full pipeline, and returns them ranked best-first (deterministically).

In [ ]:
from orme_lab.pipeline import run_screen

records = run_screen()
survivors = [r for r in records if not r.ruled_out]
print(f'{len(records)} candidates, {len(survivors)} not ruled out')

print('\nTop 8:')
for r in records[:8]:
    print(f'  {r.element:2} {r.geometry:9} {r.spin_label:9} '
          f'coup={r.coupling:.3f} plaus={r.sc_plausibility:.4f} '
          f'ruled_out={r.ruled_out}')

## The isolation gate (hypotheses 4 & 5)

Every **monomer** must be ruled out: an electronically isolated atom has no channel for a macroscopic coherent condensate. If a monomer ever survived, that would be a model bug, not a discovery.

In [ ]:
monomers = [r for r in records if r.geometry == 'monomer']
assert all(r.ruled_out for r in monomers), 'BUG: a monomer survived the SC gate!'
print('OK: all', len(monomers), 'monomers ruled out (isolation gate holds).')

## Write the ranked CSV

The deliverable: a ranked CSV of candidate states in `outputs/`.

In [ ]:
from orme_lab.pipeline import write_csv
os.makedirs('../outputs', exist_ok=True)
path = write_csv(records, '../outputs/overview_screen.csv')
print('wrote', path)